In [1]:
# Bronze Layer Validation Script
import os
import pandas as pd
from src.database import get_engine

In [ ]:
# Load RDS credentials from environment variables
RDS_HOST = os.getenv("RDS_HOST")
RDS_PORT = int(os.getenv("RDS_PORT", "5432"))
RDS_DATABASE = os.getenv("RDS_DATABASE")
RDS_USER = os.getenv("RDS_USER")
RDS_PASSWORD = os.getenv("RDS_PASSWORD")

BRONZE_SCHEMA = os.getenv("BRONZE_SCHEMA", "bronze_group6")

# Create SQLAlchemy engine 
engine = get_engine()

In [3]:
def run_query(query: str, description: str):
    print(f"\n {description}")
    df = pd.read_sql(query, engine)
    print(df)
    return df

In [4]:
# List all tables in bronze schema
tables_query = f"""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = '{BRONZE_SCHEMA}'
ORDER BY table_name;
"""
tables_df = run_query(tables_query, "Tables in Bronze Schema")


 Tables in Bronze Schema
         table_name
0       clickstream
1  order_line_items
2            orders
3          products
4           reviews
5             users


In [6]:
# Check table count
num_tables = len(tables_df)
print(f"\n Number of tables: {num_tables}")
if num_tables == 6:
    print(" Expected 6 tables in Bronze schema!")
else:
    print(" Unexpected number of tables")


 Number of tables: 6
 Expected 6 tables in Bronze schema!


In [7]:
# Row counts per table
counts_query = f"""
SELECT 'products' AS table_name, COUNT(*) AS rows FROM {BRONZE_SCHEMA}.products
UNION ALL
SELECT 'users', COUNT(*) FROM {BRONZE_SCHEMA}.users
UNION ALL
SELECT 'orders', COUNT(*) FROM {BRONZE_SCHEMA}.orders
UNION ALL
SELECT 'order_line_items', COUNT(*) FROM {BRONZE_SCHEMA}.order_line_items
UNION ALL
SELECT 'reviews', COUNT(*) FROM {BRONZE_SCHEMA}.reviews
UNION ALL
SELECT 'clickstream', COUNT(*) FROM {BRONZE_SCHEMA}.clickstream;
"""
counts_df = run_query(counts_query, "Row counts per table")



 Row counts per table
         table_name    rows
0          products     228
1             users    5000
2            orders   17072
3  order_line_items   30884
4           reviews    2930
5       clickstream  544041


In [8]:
# Inspect columns of products table
columns_query = f"""
SELECT column_name
FROM information_schema.columns
WHERE table_schema = '{BRONZE_SCHEMA}' AND table_name = 'products'
ORDER BY ordinal_position;
"""
columns_df = run_query(columns_query, "Columns of products table")
print(f"\n Products table has {len(columns_df)} columns (expected ~21)")


 Columns of products table
             column_name
0             product_id
1           product_uuid
2                  brand
3             model_name
4               colorway
5               category
6            subcategory
7                   slug
8           display_name
9              price_usd
10    _internal_cost_usd
11          _supplier_id
12   _warehouse_location
13   _internal_cost_code
14  available_sizes_json
15            image_urls
16          weight_grams
17             is_active
18       is_hype_product
19            created_at
20                  tags

 Products table has 21 columns (expected ~21)


In [9]:
#  Order statuses
status_query = f"""
SELECT status, COUNT(*) AS cnt
FROM {BRONZE_SCHEMA}.orders
GROUP BY status
ORDER BY cnt DESC;
"""
status_df = run_query(status_query, "Order statuses")
print("\n Expected statuses: delivered, shipped, processing, returned, cancelled, chargeback")


 Order statuses
       status    cnt
0   delivered  12317
1     shipped   1994
2    returned   1042
3  processing    830
4   cancelled    530
5  chargeback    359

 Expected statuses: delivered, shipped, processing, returned, cancelled, chargeback


In [10]:
#  Review ratings
ratings_query = f"""
SELECT rating, COUNT(*) AS cnt
FROM {BRONZE_SCHEMA}.reviews
GROUP BY rating
ORDER BY rating;
"""
ratings_df = run_query(ratings_query, "Review ratings")
print("\n Expected ratings: 1–5 (integer)")


 Review ratings
   rating   cnt
0       1   153
1       2   241
2       3   359
3       4   912
4       5  1265

 Expected ratings: 1–5 (integer)


In [12]:
# Clickstream event types
events_query = f"""
SELECT event_type, COUNT(*) AS cnt
FROM {BRONZE_SCHEMA}.clickstream
GROUP BY event_type
ORDER BY cnt DESC;
"""
events_df = run_query(events_query, "Clickstream event types")
print("\n Expected event types: page_view, product_view, add_to_cart, checkout")



 Clickstream event types
  event_type     cnt
0   pageview  544041

 Expected event types: page_view, product_view, add_to_cart, checkout


In [13]:
# Final Checkpoint
if num_tables == 6 and all(counts_df['rows'] > 0):
    print("\n Final Checkpoint: 6 populated tables in Bronze schema")
else:
    print("\n Checkpoint incomplete: verify table counts and data")


 Final Checkpoint: 6 populated tables in Bronze schema
